In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

In [6]:
CANDIDATES = [Path("../data/imputation"), Path(".")]
DATA_DIR = next(p for p in CANDIDATES if (p / "stage2_train_df_filled.csv").exists())
print(DATA_DIR)

..\data\imputation


In [7]:

train = pd.read_csv(DATA_DIR / "stage2_train_df_filled.csv")
val   = pd.read_csv(DATA_DIR / "stage2_val_df_filled.csv")
print("train", train.shape, "| val", val.shape)


for key in ["ad_id", "title_group"]:
    print(f"{key:12s} overlap train/val:", len(set(train[key]) & set(val[key])))


print("price range train:", train.price.min(), "-", train.price.max(), "| val:", val.price.min(), "-", val.price.max())


train (5165, 54) | val (1252, 54)
ad_id        overlap train/val: 0
title_group  overlap train/val: 0
price range train: 1000.0 - 152000.0 | val: 1000.0 - 185500.0


عشان نحدد امتى السعر يستحق يتراجع

In [19]:
MIN_N        = 3     # minimum number of OTHER ads for a model_key reference  (defined by repeating)
RATIO_LOW    = 0.40   # price below 40% of the model median -> review
RATIO_HIGH   = 2.50   # price above 250% of the model median -> review
MAD_Z_LIMIT  = 3.5    # Iglewicz-Hoaglin modified z-score limit
MAD_FLOOR    = 0.10   # floor for MAD on log scale (ties / round prices make MAD tiny)
PLACEHOLDER_MAX = 1000  # platform minimum price -> treated as a placeholder

Build a price reference for each known phone model using training data.


In [20]:
def build_reference(train):
    
    known_ads = train[train["model_key"] != "unknown"].copy()
    known_ads["log_price"] = np.log(known_ads["price"])  #to reduce the effect of very large prices
    model_prices = known_ads.groupby("model_key")["log_price"]

    reference = pd.DataFrame({
        "n": model_prices.size(),
        "med_log": model_prices.median()
    })

    # Calculate how much prices usually differ from the model's median
    price_difference = ( known_ads["log_price"] - known_ads["model_key"].map(reference["med_log"])).abs()

    known_ads["price_difference"] = price_difference

    mad_by_model = known_ads.groupby("model_key")["price_difference"].median()
    reference["mad_log"] = mad_by_model

    # Use only models with enough ads
    reference = reference[reference["n"] >= MIN_N]

    return reference

ref = build_reference(train)
print("Number of models:", len(ref))
display(ref.head(10))

Number of models: 286


,n,med_log,mad_log
model_key,,,
apple iphone 11,242,9.350102,0.130236
apple iphone 11 pro,68,9.472705,0.143101
apple iphone 11 pro max,57,9.711116,0.115513
apple iphone 12,93,9.546813,0.114603
apple iphone 12 mini,15,9.392662,0.080043
apple iphone 12 pro,44,9.798127,0.105361
apple iphone 12 pro max,13,10.085809,0.087011
apple iphone 13,372,9.998798,0.146603
apple iphone 13 mini,8,9.615805,0.139657


Calculate a reference for each ad using other ads of the same model


In [21]:

def get_train_references(df):

    log_prices = np.log(df["price"].to_numpy())
    references = np.full((len(df), 3), np.nan)

    positions = {idx: pos for pos, idx in enumerate(df.index)}

    for model, indices in df.groupby("model_key").groups.items():
        if model == "unknown":
            continue

        model_positions = np.array([positions[i] for i in indices])

        for pos in model_positions:
            other_prices = log_prices[model_positions[model_positions != pos]]

            if len(other_prices) < MIN_N:
                continue

            median = np.median(other_prices)
            mad = np.median(np.abs(other_prices - median))

            references[pos] = [len(other_prices), median, mad]

    return references


In [22]:

references = get_train_references(train)

reference_results = pd.DataFrame(
    references,
    columns=["reference_count", "median_log_price", "mad_log"],
    index=train.index
)

reference_results["model_key"] = train["model_key"]
reference_results["actual_price"] = train["price"].values
reference_results["reference_median_price"] = np.exp(
    reference_results["median_log_price"]
).round(0)

display(reference_results.head(10))


,reference_count,median_log_price,mad_log,model_key,actual_price,reference_median_price
0,371.0,9.998798,0.146603,apple iphone 13,19500.0,22000.0
1,5.0,11.034890,0.016000,honor magic v5,80000.0,62000.0
2,NaN,NaN,NaN,oppo find x9 pro,57999.0,NaN
3,NaN,NaN,NaN,honor robot,94000.0,NaN
4,NaN,NaN,NaN,xiaomi 17 ultra,70000.0,NaN
5,34.0,10.703181,0.040074,apple iphone 15 pro,60000.0,44497.0
6,241.0,9.350102,0.122602,apple iphone 11,10000.0,11500.0
7,101.0,10.839581,0.075508,apple iphone 15 pro max,58000.0,51000.0
8,6.0,9.865182,0.037993,huawei nova 15max,19000.0,19248.0
9,NaN,NaN,NaN,apple iphone 16e,36000.0,NaN


In [23]:
has_reference = reference_results["reference_count"].notna()

print("Total training ads:", len(reference_results))
print("Ads with reference:", has_reference.sum())
print("Ads without reference:", (~has_reference).sum())
print("Coverage (%):", round(has_reference.mean() * 100, 2))

Total training ads: 5165
Ads with reference: 3967
Ads without reference: 1198
Coverage (%): 76.81


In [25]:

unknown_mask = train["model_key"].eq("unknown")

model_counts = train["model_key"].value_counts()
ads_per_model = train["model_key"].map(model_counts).fillna(0)

print("Total ads:", len(train))
print("Unknown model ads:", unknown_mask.sum())

for min_n in [3]:
    has_reference = (
        ~unknown_mask
        & (ads_per_model - 1 >= min_n)
    )

    no_reference = ~has_reference

    known_but_few_ads = no_reference & ~unknown_mask

    print(f"\nMIN_N = {min_n}")
    print("Ads with reference:", has_reference.sum())
    print("Unknown model ads:", unknown_mask.sum())
    print("Known model but insufficient ads:", known_but_few_ads.sum())
    print("Total without reference:", no_reference.sum())


Total ads: 5165
Unknown model ads: 465

MIN_N = 3
Ads with reference: 3967
Unknown model ads: 465
Known model but insufficient ads: 733
Total without reference: 1198


-------------------------------------------------

------------------------------------------------------

for price reason review

In [30]:
import re
RX_TEST   = re.compile(r"pw\s*mobile\s*phone\s*test", re.I)
RX_PARTS  = re.compile(r"^\s*(?:شاش[هة]|بطاري[هة]|جراب|شاحن|سماع[هة]|اسكرين[هة]|كفر|بورد[هة])|touch\s*screen\s*panel|قطع\s*غيار|spare\s*parts", re.I)
RX_WANTED = re.compile(r"لو عندك|عاوز اشتري|عايز اشتري|مطلوب|نشتري|محتاج اشتري|wanted", re.I)

Compare ad prices and flag ads that need review.


In [31]:

def score_prices(df, ref=None, loo=False):
    d = df.copy()

    if loo:
        r = get_train_references(d)
        d[["ref_n", "ref_med_log", "ref_mad_log"]] = r
    else:
        for col in ["n", "med_log", "mad_log"]:
            d[{"n": "ref_n", "med_log": "ref_med_log",
               "mad_log": "ref_mad_log"}[col]] = d["model_key"].map(ref[col])

    has_ref = d["ref_med_log"].notna()
    ref_price = np.exp(d["ref_med_log"])

    d["ref_level"] = np.where(has_ref, "model_key", "none")
    d["ref_median_price"] = ref_price.round(0)
    d["price_ratio"] = d["price"] / ref_price
    d["mad_z"] = 0.6745 * (np.log(d["price"]) - d["ref_med_log"]) / np.maximum(d["ref_mad_log"], MAD_FLOOR)

    title = d["title"].fillna("").astype(str)
    url = d["ad_url"].fillna("").astype(str)

    reasons = pd.DataFrame({
        "price_placeholder": d["price"] <= PLACEHOLDER_MAX,
        "test_ad": title.str.contains(RX_TEST) | url.str.contains("test-edited", case=False),
        "parts_or_accessory": title.str.contains(RX_PARTS),
        "wanted_ad": title.str.contains(RX_WANTED),
        "ratio_low": has_ref & (d["price_ratio"] < RATIO_LOW),
        "ratio_high": has_ref & (d["price_ratio"] > RATIO_HIGH),
        "mad_low": has_ref & (d["mad_z"] < -MAD_Z_LIMIT),
        "mad_high": has_ref & (d["mad_z"] > MAD_Z_LIMIT),
    }, index=d.index)

    d["price_review_reason"] = reasons.apply(
        lambda row: ";".join(row.index[row]), axis=1
    )
    d["price_flag"] = d["price_review_reason"].ne("")

    context = pd.DataFrame({
        "exchange": d["mentions_exchange"].fillna(False).astype(bool),
        "bundle": d["bundle_hint"].fillna(False).astype(bool),
    }, index=d.index)

    d["price_context"] = context.apply(
        lambda row: ";".join(row.index[row]), axis=1
    )

    return d


In [32]:
train_s = score_prices(train, loo=True)        # leave-one-out reference
val_s   = score_prices(val, ref=ref)           # reference from TRAIN only
print("model_keys with a usable reference:", len(ref), "of", train.model_key.nunique())
rows = []
for name, d in [("train", train_s), ("val", val_s)]:
    rows.append({"split": name, "rows": len(d), "with_reference": int((d.ref_level == "model_key").sum()),
                 "with_reference_%": round((d.ref_level == "model_key").mean() * 100, 1),
                 "flagged": int(d.price_flag.sum()), "flagged_%": round(d.price_flag.mean() * 100, 2)})
display(pd.DataFrame(rows))

model_keys with a usable reference: 286 of 723


,split,rows,with_reference,with_reference_%,flagged,flagged_%
0,train,5165,3967,76.8,191,3.70
1,val,1252,850,67.9,49,3.91


In [33]:
for name, d in [("train", train_s), ("val", val_s)]:
    print(f"\n{name}")

    print(
        d["price_review_reason"]
        .str.split(";")
        .explode()
        .loc[lambda s: s.ne("")]
        .value_counts()
    )


train
price_review_reason
mad_low               130
ratio_low              90
price_placeholder      27
mad_high               21
ratio_high              8
parts_or_accessory      6
wanted_ad               4
test_ad                 2
Name: count, dtype: int64

val
price_review_reason
mad_low              34
ratio_low            18
mad_high              8
price_placeholder     4
ratio_high            4
test_ad               1
Name: count, dtype: int64


In [36]:
cols = [
    "title",
    "price",
    "model_key",
    "ref_median_price",
    "price_ratio",
    "mad_z",
    "price_review_reason"
]

print("TRAIN")
display(train_s[cols].round(3).head(20))

print("VALIDATION")
display(val_s[cols].round(3).head(20))

TRAIN


,title,price,model_key,ref_median_price,price_ratio,mad_z,price_review_reason
0,ايفون 13 iPhone,19500.0,apple iphone 13,22000.0,0.886,-0.555,
1,Honor Magic V5,80000.0,honor magic v5,62000.0,1.290,1.719,
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,57999.0,oppo find x9 pro,NaN,NaN,NaN,
3,Honor Robot Phone 512G 12G ram,94000.0,honor robot,NaN,NaN,NaN,
4,شاومي 17 الترا xiaomi 17 ultra,70000.0,xiaomi 17 ultra,NaN,NaN,NaN,
5,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,60000.0,apple iphone 15 pro,44497.0,1.348,2.016,
6,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,10000.0,apple iphone 11,11500.0,0.870,-0.769,
7,آبل - آيفون 15 برو ماكس,58000.0,apple iphone 15 pro max,51000.0,1.137,0.868,
8,Huawei nova 15 max,19000.0,huawei nova 15max,19248.0,0.987,-0.088,
9,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iPh...,36000.0,apple iphone 16e,NaN,NaN,NaN,


VALIDATION


,title,price,model_key,ref_median_price,price_ratio,mad_z,price_review_reason
0,Honor 500 pro,35000.0,honor 500 pro,NaN,NaN,NaN,
1,آبل - آيفون 13 برو ماكس,38000.0,apple iphone 13 pro max,28000.0,1.357,1.079,
2,سامسونج اس 24 الترا,40000.0,samsung s24 ultra,47000.0,0.851,-1.088,
3,Honor win rt,30500.0,honor win rt,NaN,NaN,NaN,
4,Iphone 13 pro max,37000.0,apple iphone 13 pro max,28000.0,1.321,0.985,
5,سامسونج اس 24 الترا,45000.0,samsung s24 ultra,47000.0,0.957,-0.293,
6,Apple - iPhone 17 Pro Max - Battery 98%,92000.0,apple iphone 17 pro max,83000.0,1.108,0.694,
7,Galaxy Z TriFold,149999.0,samsung z trifold,NaN,NaN,NaN,
8,Vivo V60,27000.0,vivo v60,20000.0,1.350,1.246,
9,سامسونج اس 25 الترا,54000.0,samsung s25 ultra,55000.0,0.982,-0.124,


In [ ]:
print("Flagged TRAIN")
display(train_s.loc[train_s["price_flag"], cols].round(3).head(20))

print("Flagged VALIDATION")
display(val_s.loc[val_s["price_flag"], cols].round(3).head(20))

Flagged TRAIN


,title,price,model_key,ref_median_price,price_ratio,mad_z,price_review_reason
54,اوبو رينو 12 اف,6500.0,oppo reno 12f,11500.0,0.565,-3.737,mad_low
96,Apple - iPhone 17 Pro Max First high copy,4500.0,apple iphone 17 pro max,83000.0,0.054,-19.660,ratio_low;mad_low
161,آبل - آيفون 17 برو ماكس,4500.0,apple iphone 17 pro max,83000.0,0.054,-19.660,ratio_low;mad_low
208,سامسونج اس 21 الترا هاي كوبي,5000.0,samsung s21 ultra,15199.0,0.329,-5.045,ratio_low;mad_low
219,آبل - آيفون 12,6500.0,apple iphone 12,14000.0,0.464,-4.541,mad_low
235,شاشه touchscreen panel sony xperia Z Ultra للبيع,1500.0,sony xperia z ultra,NaN,NaN,NaN,parts_or_accessory
240,آبل - آيفون 11,2000.0,apple iphone 11,11500.0,0.174,-9.623,ratio_low;mad_low
277,آبل - آيفون 12,6500.0,apple iphone 12,14000.0,0.464,-4.541,mad_low
279,هونر 6 اكس,1500.0,honor x6,9500.0,0.158,-12.450,ratio_low;mad_low
292,موبيلات صيانه للبيع,1000.0,unknown,NaN,NaN,NaN,price_placeholder


Flagged VALIDATION


,title,price,model_key,ref_median_price,price_ratio,mad_z,price_review_reason
64,آبل - آيفون 14,48000.0,apple iphone 14,28000.0,1.714,3.636,mad_high
87,Apple - iPhone 8,2800.0,apple iphone 8,5800.0,0.483,-4.912,mad_low
93,شاومي بوكو اكس 3 برو,1500.0,xiaomi poco x3 pro,4599.0,0.326,-7.557,ratio_low;mad_low
119,Airbudspro,1000.0,unknown,NaN,NaN,NaN,price_placeholder
172,آبل - آيفون 18 برو ماكس,6000.0,apple iphone 18 pro max,110000.0,0.055,-13.383,ratio_low;mad_low
216,ايفون 8 كسر زيرو حاله نضيفه,3000.0,apple iphone 8,5800.0,0.517,-4.447,mad_low
263,المدارس فتحت ووقت التغيير جه مع سامسونج S26 الترا,2500.0,samsung s26 ultra,67000.0,0.037,-22.180,ratio_low;mad_low
264,Redmi Not 10 Pro,1900.0,xiaomi redmi note 10 pro,5848.0,0.325,-4.507,ratio_low;mad_low
269,سامسونج ايه 71,3100.0,samsung a71,7000.0,0.443,-5.494,mad_low
282,PW Mobile Phone Test 1789631680715,10000.0,unknown,NaN,NaN,NaN,test_ad


In [45]:

output_dir = Path("../data/processed/price_review_task4")
output_dir.mkdir(parents=True, exist_ok=True)

train_s.to_csv(
    output_dir / "train_scored_4_1.csv",
    index=False,
    encoding="utf-8-sig"
)

val_s.to_csv(
    output_dir / "val_scored_4_1.csv",
    index=False,
    encoding="utf-8-sig"
)